In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/gold-standard-recheck/Gold_label_dataset_aspect_and_unlabel_recheck.csv


# **Hyperparameter Tuning**

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, ShuffleSplit
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
import nltk
from transformers import RobertaTokenizer, RobertaForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset
import torch
import optuna
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os
import json

warnings.filterwarnings("ignore")
print("Downloading NLTK data...")
nltk.download('punkt', quiet=True)


np.random.seed(42)
torch.manual_seed(42)

best_model_dir = './Best_Roberta_Model'
os.makedirs(best_model_dir, exist_ok=True)

In [3]:
print("Downloading NLTK data...")
nltk.download('punkt', quiet=True)

print("Loading data...")
df = pd.read_csv('/kaggle/input/gold-standard-recheck/Gold_label_dataset_aspect_and_unlabel_recheck.csv')
df = df.rename(columns={'Sentence': 'sentence', 'Label': 'label'})



Loading data...


In [4]:
# Check distribution of dataset and choose only rows that contain aspect annotation
labeled_data = df[df['label'].notna()]
print("Total samples:", len(df))
print("Labeled samples:", len(labeled_data))
print("Unlabeled samples:", len(df) - len(labeled_data))
print("\nInitial label distribution:")
print(labeled_data['label'].value_counts())

Total samples: 204622
Labeled samples: 1175
Unlabeled samples: 203447

Initial label distribution:
label
Design & Material        194
Instructor               187
Assignment & Tutorial    178
Ambiguous                159
Platform & Technical     156
Skill & Career           151
Financial                150
Name: count, dtype: int64


In [5]:

label_encoder = LabelEncoder()
labeled_data['label'] = label_encoder.fit_transform(labeled_data['label'])
label_mapping = dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)))
print("\nLabel mapping:", label_mapping)


Label mapping: {'Ambiguous': 0, 'Assignment & Tutorial': 1, 'Design & Material': 2, 'Financial': 3, 'Instructor': 4, 'Platform & Technical': 5, 'Skill & Career': 6}


In [6]:
train_data, test_data = train_test_split(labeled_data, test_size=0.2, random_state=42, stratify=labeled_data['label'])
print("Train set size:", len(train_data))
print("Test set size:", len(test_data))

Train set size: 940
Test set size: 235


In [7]:
tokenizer = RobertaTokenizer.from_pretrained("roberta-base")
def tokenize_function(examples, max_length):
    return tokenizer(examples['sentence'], padding='max_length', truncation=True, max_length=max_length)

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

In [8]:
# Computing metrics
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='weighted', zero_division=1)
    acc = accuracy_score(labels, preds)
    
    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

In [9]:
def objective(trial, dataset, trial_number):
    params = {
        'learning_rate': trial.suggest_float('learning_rate', 1e-5, 1e-3, log=True),
        'per_device_train_batch_size': trial.suggest_categorical('per_device_train_batch_size', [32, 64]),
        'weight_decay': trial.suggest_float('weight_decay', 1e-5, 1e-1, log=True),
        'max_length': trial.suggest_categorical('max_length', [64, 128])
    }
    
    print(f"\nTrial {trial_number}")
    print(f"Hyperparameters:")
    for key, value in params.items():
        print(f"  {key}: {value}")
    
    n_splits = 2
    test_size = 0.2
    ss = ShuffleSplit(n_splits=n_splits, test_size=test_size, random_state=42)
    best_f1_scores = []
    
    for fold, (train_index, val_index) in enumerate(ss.split(dataset), 1):
        print(f"\nFold {fold}:")
        train_fold = dataset.select(train_index)
        val_fold = dataset.select(val_index)
        
        print(f"  Train size: {len(train_fold)}")
        print(f"  Val size: {len(val_fold)}")
        
        tokenized_train = train_fold.map(lambda x: tokenize_function(x, params['max_length']), batched=True)
        tokenized_val = val_fold.map(lambda x: tokenize_function(x, params['max_length']), batched=True)
        
        model = RobertaForSequenceClassification.from_pretrained("roberta-base", num_labels=len(label_mapping))
        
        training_args = TrainingArguments(
            output_dir=f'./results_trial_{trial_number}_fold_{fold}',
            num_train_epochs=30,  # Reduced for faster hyperparameter tuning
            per_device_train_batch_size=params['per_device_train_batch_size'],
            per_device_eval_batch_size=params['per_device_train_batch_size'],
            weight_decay=params['weight_decay'],
            learning_rate=params['learning_rate'],
            evaluation_strategy="epoch",
            save_strategy="no",
            load_best_model_at_end=False,
            metric_for_best_model="f1",
            report_to="none"
        )
        
        trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=tokenized_train,
            eval_dataset=tokenized_val,
            compute_metrics=compute_metrics,
        )
        
        trainer.train()
        eval_result = trainer.evaluate()
        best_f1_scores.append(eval_result['eval_f1'])
        print(f"  Fold {fold} F1 score: {eval_result['eval_f1']:.4f}")
    
    mean_f1_score = np.mean(best_f1_scores)
    print(f"Mean F1 score across folds: {mean_f1_score:.4f}")
    
    return mean_f1_score

In [10]:
# starting hyperparameter tuning
train_dataset = Dataset.from_pandas(train_data)
study = optuna.create_study(direction="maximize")
study.optimize(lambda trial: objective(trial, train_dataset, trial.number), n_trials=15)

[I 2024-08-22 13:40:22,998] A new study created in memory with name: no-name-622334da-72f5-432d-82aa-6402cc08b6c1



Trial 0
Hyperparameters:
  learning_rate: 0.0002891415833033963
  per_device_train_batch_size: 64
  weight_decay: 0.00016802021126553177
  max_length: 64

Fold 1:
  Train size: 752
  Val size: 188


Map:   0%|          | 0/752 [00:00<?, ? examples/s]

Map:   0%|          | 0/188 [00:00<?, ? examples/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


... (158 outputs left out)

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,No log,1.906277,0.234043,0.106594,0.776709,0.234043
2,No log,1.529930,0.510638,0.479466,0.642162,0.510638
3,No log,1.113153,0.643617,0.616775,0.674509,0.643617
4,No log,0.847914,0.739362,0.719130,0.744480,0.739362
5,No log,0.715827,0.765957,0.759840,0.774054,0.765957
6,No log,0.672237,0.792553,0.787017,0.795809,0.792553
7,No log,0.625073,0.808511,0.806936,0.810650,0.808511
8,No log,0.653433,0.808511,0.805165,0.813354,0.808511
9,No log,0.646315,0.797872,0.796339,0.797197,0.797872
10,No log,0.730891,0.797872,0.792436,0.801998,0.797872


[I 2024-08-22 15:28:45,361] Trial 14 finished with value: 0.8419422074582601 and parameters: {'learning_rate': 3.888160537937402e-05, 'per_device_train_batch_size': 64, 'weight_decay': 9.844676365342518e-05, 'max_length': 128}. Best is trial 0 with value: 0.8655893666270169.


  Fold 2 F1 score: 0.8454
Mean F1 score across folds: 0.8419


In [11]:
# Save the best hyperparameter set in json form in order to reuse in the next process
print("\nBest trial:")
trial = study.best_trial
print("Value (Mean F1 score): ", trial.value)
print("Params: ")
for key, value in trial.params.items():
    print("    {}: {}".format(key, value))

best_hyperparameters = study.best_params


Best trial:
Value (Mean F1 score):  0.8655893666270169
Params: 
    learning_rate: 0.0002891415833033963
    per_device_train_batch_size: 64
    weight_decay: 0.00016802021126553177
    max_length: 64


In [12]:
with open('best_hyperparameters.json', 'w') as f:
    json.dump(best_hyperparameters, f)
print("Best hyperparameters saved to best_hyperparameters.json")

Best hyperparameters saved to best_hyperparameters.json


# **Fine-tuning Roberta on the best hyperparameter sets**

In [13]:
def train_model(train_dataset, test_dataset, output_dir, num_epochs):
    model = RobertaForSequenceClassification.from_pretrained("roberta-base", num_labels=len(label_mapping))
    
    training_args = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=num_epochs,
        per_device_train_batch_size=best_hyperparameters['per_device_train_batch_size'],
        per_device_eval_batch_size=best_hyperparameters['per_device_train_batch_size'],
        weight_decay=best_hyperparameters['weight_decay'],
        learning_rate=best_hyperparameters['learning_rate'],
        evaluation_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        save_total_limit=1,
        report_to="none"
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=test_dataset,
        compute_metrics=compute_metrics,
    )

    print(f"Training model for {num_epochs} epochs...")
    trainer.train()
    
    return trainer, trainer.state.best_metric


In [14]:
train_dataset = train_dataset.map(lambda x: tokenize_function(x, best_hyperparameters['max_length']), batched=True)
test_dataset = Dataset.from_pandas(test_data).map(lambda x: tokenize_function(x, best_hyperparameters['max_length']), batched=True)

Map:   0%|          | 0/940 [00:00<?, ? examples/s]

Map:   0%|          | 0/235 [00:00<?, ? examples/s]

In [15]:
# Trying with different Epoch Sizes
epochs_to_try = [30, 40]
results = []

for epochs in epochs_to_try:
    print(f"\nTraining model for {epochs} epochs...")
    trainer, best_f1 = train_model(train_dataset, test_dataset, f"./model_epochs_{epochs}", epochs)
    results.append((epochs, best_f1))
    print(f"Best F1 score for {epochs} epochs: {best_f1}")


Training model for 30 epochs...


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Training model for 30 epochs...


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,No log,1.110500,0.663830,0.679131,0.758599,0.663830
2,No log,0.664292,0.825532,0.816599,0.849125,0.825532
3,No log,0.540385,0.863830,0.862952,0.865355,0.863830
4,No log,0.507899,0.885106,0.883282,0.885023,0.885106
5,No log,0.762975,0.834043,0.825505,0.852546,0.834043
6,No log,0.679644,0.859574,0.855109,0.863517,0.859574
7,No log,0.617310,0.838298,0.838768,0.860622,0.838298
8,No log,0.498618,0.893617,0.890428,0.895431,0.893617
9,No log,0.587228,0.893617,0.892502,0.899833,0.893617
10,No log,0.690903,0.876596,0.873062,0.884754,0.876596


Best F1 score for 30 epochs: 0.9049726518560685

Training model for 40 epochs...


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Training model for 40 epochs...


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,No log,1.548286,0.357447,0.290257,0.655651,0.357447
2,No log,0.969499,0.693617,0.675688,0.742802,0.693617
3,No log,0.591365,0.859574,0.857651,0.872571,0.859574
4,No log,0.599932,0.838298,0.831752,0.846843,0.838298
5,No log,0.733347,0.851064,0.842559,0.865759,0.851064
6,No log,0.622166,0.868085,0.863841,0.877127,0.868085
7,No log,0.911815,0.791489,0.791070,0.823553,0.791489
8,No log,0.678794,0.880851,0.880970,0.886695,0.880851
9,No log,0.705841,0.846809,0.843964,0.861653,0.846809
10,No log,0.404277,0.914894,0.913450,0.916353,0.914894


Best F1 score for 40 epochs: 0.9142405689913001


In [16]:
# Show results
print("\nSummary of results:")
for epochs, f1_score in results:
    print(f"Epochs: {epochs}, Best F1 score: {f1_score}")

# Find best number of epochs
best_epochs, best_f1 = max(results, key=lambda x: x[1])
print(f"\nBest performance: {best_epochs} epochs with F1 score of {best_f1}")


Summary of results:
Epochs: 30, Best F1 score: 0.9049726518560685
Epochs: 40, Best F1 score: 0.9142405689913001

Best performance: 40 epochs with F1 score of 0.9142405689913001


In [17]:
# Retrain with best number of epochs
print(f"\nRetraining model with best epochs: {best_epochs}")
final_trainer, final_f1 = train_model(train_dataset, test_dataset, best_model_dir, best_epochs)

final_trainer.save_model(best_model_dir)
print(f"Best model saved to {best_model_dir}")

tokenizer.save_pretrained(best_model_dir)
print(f"Tokenizer saved to {best_model_dir}")

print("\nEvaluating final model on test set...")
final_results = final_trainer.evaluate(test_dataset)
print("Final evaluation results:", final_results)


Retraining model with best epochs: 40


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Training model for 40 epochs...


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,No log,1.733613,0.268085,0.216073,0.823554,0.268085
2,No log,0.874009,0.765957,0.765685,0.816014,0.765957
3,No log,0.707276,0.800000,0.786644,0.815796,0.800000
4,No log,0.524353,0.859574,0.855918,0.865388,0.859574
5,No log,0.424305,0.902128,0.899947,0.905636,0.902128
6,No log,0.581428,0.868085,0.863698,0.878993,0.868085
7,No log,0.516541,0.885106,0.882737,0.890275,0.885106
8,No log,0.558339,0.872340,0.871916,0.876709,0.872340
9,No log,0.685169,0.872340,0.872088,0.873315,0.872340
10,No log,0.581732,0.876596,0.872730,0.878749,0.876596


Best model saved to ./Best_Roberta_Model
Tokenizer saved to ./Best_Roberta_Model

Evaluating final model on test set...


Final evaluation results: {'eval_loss': 0.6130794286727905, 'eval_accuracy': 0.9106382978723404, 'eval_f1': 0.9102281653310095, 'eval_precision': 0.9160078867728402, 'eval_recall': 0.9106382978723404, 'eval_runtime': 0.5314, 'eval_samples_per_second': 442.264, 'eval_steps_per_second': 3.764, 'epoch': 40.0}


# **Create pseudo datasets**

In [18]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from transformers import RobertaTokenizer, RobertaForSequenceClassification
import torch
from sklearn.preprocessing import LabelEncoder
from tqdm.auto import tqdm
import os
import warnings
import nltk
from nltk.corpus import wordnet
import random
data = pd.read_csv('/kaggle/input/gold-standard-recheck/Gold_label_dataset_aspect_and_unlabel_recheck.csv')
data = data.rename(columns={'Sentence': 'sentence', 'Label': 'label'})
labeled_data = data[data['label'].notna()]
unlabeled_data = data[data['label'].isna()]
print(f"Total samples: {len(data)}")
print(f"Labeled samples: {len(labeled_data)}")
print(f"Unlabeled samples: {len(unlabeled_data)}")
nltk.data.path.append('/root/nltk_data')
nltk.download('wordnet', download_dir='/root/nltk_data')
nltk.download('omw-1.4', download_dir='/root/nltk_data')

!python3 -m nltk.downloader wordnet
!unzip /root/nltk_data/corpora/wordnet.zip -d /root/nltk_data/corpora/

nltk.data.path.append('/root/nltk_data')
nltk.download('wordnet', download_dir='/root/nltk_data')
nltk.download('omw-1.4', download_dir='/root/nltk_data')


Total samples: 204622
Labeled samples: 1175
Unlabeled samples: 203447
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
/opt/conda/lib/python3.10/runpy.py:126: RuntimeWarning: 'nltk.downloader' found in sys.modules after import of package 'nltk', but prior to execution of 'nltk.downloader'; this may result in unpredictable behaviour
  warn(RuntimeWarning(msg))
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
Archive:  /root/nltk_data/corpora/wordnet.zip
   creating: /root/nltk_data/corpora/wordnet/
  inflating: /root/nltk_data/corpora/wordnet/lexnames  
  inflating: /root/nltk_data/corpora/wordnet/data.verb  
  inflating: /root/nltk_data/corpora/wordnet/index.adv  
  inflating: /root/nltk_data/corpora/wordnet/adv.exc  
  inflating: /root/nltk_data/corpora/wordnet/index.verb  
  inflating: /root/nltk_data/corpora/wordnet/cntlist.rev  
  inflating: /

True

In [19]:
label_encoder = LabelEncoder()
labeled_data['encoded_label'] = label_encoder.fit_transform(labeled_data['label'])
label_mapping = dict(zip(label_encoder.classes_, range(len(label_encoder.classes_))))

print("\nLabel mapping:")
for label, index in label_mapping.items():
    print(f"{label}: {index}")


Label mapping:
Ambiguous: 0
Assignment & Tutorial: 1
Design & Material: 2
Financial: 3
Instructor: 4
Platform & Technical: 5
Skill & Career: 6


In [20]:
train_data, test_data = train_test_split(labeled_data, test_size=0.2, random_state=42, stratify=labeled_data['encoded_label'])

In [21]:
# Download the best model
tokenizer = RobertaTokenizer.from_pretrained("roberta-base")
model = RobertaForSequenceClassification.from_pretrained("./Best_Roberta_Model")
model.eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
             

In [22]:
# Pseudo-labeling function, I used the best fine-tuned Roberta model form the previous process for inferencing
def pseudo_label(df, confidence_threshold):
    pseudo_labeled = []
    valid_data = df[df['sentence'].notna()]
    for _, row in tqdm(valid_data.iterrows(), total=len(valid_data), desc=f"Pseudo-labeling (conf={confidence_threshold})"):
        inputs = tokenizer(row['sentence'], padding='max_length', truncation=True, max_length=best_hyperparameters['max_length'], return_tensors='pt')
        inputs = {k: v.to(device) for k, v in inputs.items()}
        with torch.no_grad():
            outputs = model(**inputs)
            probs = outputs.logits.softmax(dim=-1)
            confidence = probs.max().item()
            predicted_class = probs.argmax().item()
            if confidence > confidence_threshold:
                pseudo_labeled.append({
                    'sentence': row['sentence'],
                    'label': label_encoder.inverse_transform([predicted_class])[0],
                    'encoded_label': predicted_class,
                    'confidence': confidence
                })
    return pd.DataFrame(pseudo_labeled)

In [23]:
# Synnonym replacement function using Wordnet
def get_synonyms(word):
    synonyms = set()
    for syn in wordnet.synsets(word):
        for lemma in syn.lemmas():
            synonyms.add(lemma.name().replace('_', ' '))
    return list(synonyms)

def synonym_replacement(sentence, n=1):
    words = sentence.split()
    new_words = words.copy()
    random_word_list = list(set([word for word in words if word.isalnum()]))
    num_replaced = 0
    for random_word in random.sample(random_word_list, len(random_word_list)):
        synonyms = get_synonyms(random_word)
        if len(synonyms) >= 1:
            synonym = random.choice(synonyms)
            new_words = [synonym if word == random_word else word for word in new_words]
            num_replaced += 1
        if num_replaced >= n:
            break
    return ' '.join(new_words)


In [24]:
# Augmenting only the minor classes
def augment_data(df, label_counts, target_count):
    augmented_data = []
    for label, count in label_counts.items():
        if count < target_count:
            label_data = df[df['label'] == label]
            num_to_add = target_count - count
            for _ in range(num_to_add):
                idx = np.random.randint(0, len(label_data))
                row = label_data.iloc[idx]
                new_sentence = synonym_replacement(row['sentence'])
                augmented_data.append({
                    'sentence': new_sentence,
                    'label': row['label'],
                    'encoded_label': row['encoded_label']
                })
    return pd.DataFrame(augmented_data)

In [25]:
from tqdm.auto import tqdm
import random
none_count = unlabeled_data['sentence'].isna().sum()
print(f"Number of None values: {none_count}")

results = {}

for confidence in [0.98,0.99]:
    pseudo_labeled_data = pseudo_label(unlabeled_data.head(200000), confidence)
    print(pseudo_labeled_data)
    print("\nPseudo-labeled data distribution:")
    print(pseudo_labeled_data['label'].value_counts())
    results[confidence] = pseudo_labeled_data

for confidence, pseudo_data in results.items():
    for max_per_class in [300, 400, 500]:
        print(f"\nCreating dataset with confidence {confidence} and max_per_class {max_per_class}")
        selected_pseudo_data = pseudo_data.groupby('label').head(max_per_class)
        combined_data = pd.concat([train_data, selected_pseudo_data], ignore_index=True)
        print(f"Selected pseudo-labeled data distribution:{selected_pseudo_data['label'].value_counts()}")
        print(f"Combined data distribution:{combined_data['label'].value_counts()}")
        filename = f"combined_data_conf{confidence}_max{max_per_class}.csv"
        output_dir = "./combined_datasets"
        os.makedirs(output_dir, exist_ok=True)
        file_path = os.path.join(output_dir, filename)
        combined_data.to_csv(file_path, index=False)
        print(f"Total samples: {len(combined_data)}")
        print(f"Original samples: {len(train_data)}")
        print(f"Pseudo-labeled samples: {len(selected_pseudo_data)}")


label_counts = train_data['label'].value_counts()
target_count = label_counts.max()
augmented_train_data = augment_data(train_data, label_counts, target_count)
augmented_train_data = pd.concat([train_data, augmented_train_data], ignore_index=True)

print("\nAugmented train data distribution:")
print(augmented_train_data['label'].value_counts())

for confidence, pseudo_data in results.items():
    for max_per_class in [300, 400, 500]:
        print(f"\nCreating dataset with confidence {confidence} and max_per_class {max_per_class}")
        selected_pseudo_data = pseudo_data.groupby('label').head(max_per_class)
        combined_data = pd.concat([augmented_train_data, selected_pseudo_data], ignore_index=True)
        print(f"Selected pseudo-labeled data distribution:{selected_pseudo_data['label'].value_counts()}")
        print(f"Combined data distribution:{combined_data['label'].value_counts()}")
        filename = f"combined_data_aug_conf{confidence}_max{max_per_class}.csv"
        output_dir = "./combined_datasets_with_aug"
        os.makedirs(output_dir, exist_ok=True)
        file_path = os.path.join(output_dir, filename)
        combined_data.to_csv(file_path, index=False)
        print(f"Total samples: {len(combined_data)}")
        print(f"Original samples: {len(train_data)}")
        print(f"Augmented samples: {len(augmented_train_data) - len(train_data)}")
        print(f"Pseudo-labeled samples: {len(selected_pseudo_data)}")


Number of None values: 0


Pseudo-labeling (conf=0.98):   0%|          | 0/200000 [00:00<?, ?it/s]

                                                 sentence  \
0       I'm starting to do Engagement Project which is...   
1         Great course Emma lots of good practical ideas.   
2       I would recommend this course as an introducto...   
3       Also should identified that its sufficient for...   
4       The course not only provides clear takeaway po...   
...                                                   ...   
186500                    Lovely yin course and sequence.   
186501  I often come back to doing it and feels great ...   
186502  I had the feeling she was/is more interested i...   
186503  Sorry but I would truly like my money back I d...   
186504                                         Sorryagain   

                        label  encoded_label  confidence  
0                   Ambiguous              0    0.997776  
1       Assignment & Tutorial              1    0.991950  
2              Skill & Career              6    0.998790  
3       Assignment & Tutorial  

Pseudo-labeling (conf=0.99):   0%|          | 0/200000 [00:00<?, ?it/s]

                                                 sentence  \
0       I'm starting to do Engagement Project which is...   
1         Great course Emma lots of good practical ideas.   
2       I would recommend this course as an introducto...   
3       Also should identified that its sufficient for...   
4       The course not only provides clear takeaway po...   
...                                                   ...   
181672                    Lovely yin course and sequence.   
181673  I often come back to doing it and feels great ...   
181674  I had the feeling she was/is more interested i...   
181675  Sorry but I would truly like my money back I d...   
181676                                         Sorryagain   

                        label  encoded_label  confidence  
0                   Ambiguous              0    0.997776  
1       Assignment & Tutorial              1    0.991950  
2              Skill & Career              6    0.998790  
3       Assignment & Tutorial  